In [1]:
import joblib
import cloudpickle
import pandas as pd
import numpy as np
from tqdm import tqdm

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import log_loss, accuracy_score, roc_auc_score

import pymc as pm
import pytensor.tensor as pt
import arviz as az
import arviz_plots as azp
import xarray as xr
from scipy.special import expit

import matplotlib.pyplot as plt
from matplotlib.patches import Arc

RANDOM_SEED = 694973
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_columns', None)

In [2]:
print(pm.__version__)

5.25.1


In [3]:
bart_mm_scaler = joblib.load('minmax_scaler.pkl')

In [4]:
df=pd.read_csv('input/full_results.csv')
df['shot_type_idx'] = df['shot_type'].map({'JUMPER': 0, 'LAYUP': 1, 'HOOK': 2, 'DUNK': 3})
df['sub_type_idx'] = df['subType_idx'] - 1
df['season_idx'] = df['season_idx'] - 1
df = df.loc[df['period']<=4].reset_index(drop=True)
df['PLAYER_ID'] = pd.to_numeric(df['PLAYER_ID'], errors='coerce')
df = df.dropna(subset=['PLAYER_ID'])
df.head()

,gameId,event_id,actionId,actionNumber,description,is_made,SHOT_DISTANCE,ANGLE,xLegacy,yLegacy,shotValue,subType,subType_idx,shot_type,shot_type_idx,SEASON,season_idx,PLAYER,PLAYER_ID,player_age,HEIGHT_INCHES,POSITION,position_idx,homeTeamId,homeTeamTricode,court_idx,awayTeamId,awayTeamTricode,defTeamId,defTeamTricode,def_idx,score_shooter,score_opposition,score_diff,period,time_minutes,time_seconds,ishome_shooter,teamId,teamTricode,is_train,distance_mm,angle_mm,probs,baseline_logit,logit_sd,sub_type_idx
0,22200001,0,3,7,MISS Embiid 13' Turnaround Fadeaway Shot,0,12.815615,67.036227,-118,50,2,Turnaround Fadeaway shot,1,JUMPER,0,2022-23,0,Joel Embiid,203954.0,29.374401,84.0,C-F,1,1.610613e+09,BOS,1,1.610613e+09,PHI,1.610613e+09,BOS,1,0.0,0.0,0.0,1,11.633333,698.0,0,1610612755,PHI,1,0.267300,0.670484,0.428531,-0.288581,0.103031,0
1,22200001,1,7,11,Smart 13' Driving Floating Bank Jump Shot (2 PTS),1,13.200379,65.376435,120,55,2,Driving Floating Bank Jump Shot,2,JUMPER,0,2022-23,0,Marcus Smart,203935.0,29.401780,76.0,G,2,1.610613e+09,BOS,1,1.610613e+09,PHI,1.610613e+09,PHI,2,2.0,0.0,2.0,1,11.250000,675.0,1,1610612738,BOS,1,0.275325,0.653883,0.427449,-0.292989,0.101178,1
2,22200001,3,10,14,Harris Tip Layup Shot (2 PTS),1,0.000000,0.000000,0,0,2,Tip Layup Shot,4,LAYUP,1,2022-23,0,Tobias Harris,202699.0,31.041752,79.0,F,3,1.610613e+09,BOS,1,1.610613e+09,PHI,1.610613e+09,BOS,1,2.0,2.0,0.0,1,11.050000,663.0,0,1610612755,PHI,1,0.000000,0.000000,0.697259,0.841187,0.188602,3
3,22200001,4,11,15,Tatum 24' 3PT Jump Shot (3 PTS) (Smart 1 AST),1,23.711811,78.074008,-232,49,3,Jump Shot,5,JUMPER,0,2022-23,0,Jayson Tatum,1628369.0,25.409993,80.0,F-G,4,1.610613e+09,BOS,1,1.610613e+09,PHI,1.610613e+09,PHI,2,5.0,2.0,3.0,1,10.766667,646.0,1,1610612738,BOS,1,0.494566,0.780882,0.384304,-0.472442,0.098655,4
4,22200001,6,17,23,MISS Maxey 4' Driving Layup,0,3.883298,78.111342,38,8,2,Driving Layup Shot,6,LAYUP,1,2022-23,0,Tyrese Maxey,1630178.0,22.735113,74.0,G,2,1.610613e+09,BOS,1,1.610613e+09,PHI,1.610613e+09,BOS,1,2.0,5.0,-3.0,1,10.200000,612.0,0,1610612755,PHI,1,0.080995,0.781255,0.487208,-0.051393,0.135053,5


In [5]:
from typing import Union
def define_index(data: pd.DataFrame, label: str) -> Union[np.array, dict]:
    """Defines an index variable starting at 1. We add 1 so '0' can act as a placeholder for
    any global optionality
    Args:
        data (pd.DataFrame): dataframe with values to index
        label (str): column name user wishes to index in string format

    Returns:
        Union[
            np.array: indexed values
            dict: dictionary mapping the input values and their indexed values
            ]
    """
    data[label] = data[label].apply(str)
    unq_ids = data[label].unique()
    n_ids = len(unq_ids)
    lookup_dict = dict(zip(unq_ids, range(n_ids)))
    return data[label].replace(lookup_dict).values + 1, lookup_dict

In [6]:
df["player_idx"], player_lookup = define_index(df, "PLAYER_ID")
# provide an index for teamID
df["team_idx"], team_lookup = define_index(df, "teamId")
# provide an index for hometeamID (court factor)
df["court_idx"], hometeam_lookup = define_index(df, "homeTeamId")
# position
df["position_idx"], position_lookup = define_index(df, "POSITION")
# team defense
df["def_idx"], def_team_lookup = define_index(df, "defTeamId")

/tmp/ipykernel_5748/2469266612.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return data[label].replace(lookup_dict).values + 1, lookup_dict
/tmp/ipykernel_5748/2469266612.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return data[label].replace(lookup_dict).values + 1, lookup_dict
/tmp/ipykernel_5748/2469266612.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the f

In [7]:
target=df['is_made'].values
df.shape

(497034, 49)

In [8]:
def sample(
    model: pm.Model, draws: int = 2000, tune: int = 2000, chains: int = 4, target_accept: float = 0.99, random_seed: int = RANDOM_SEED,
    path = 'temp.nc'
):
    """
    Fit model using MCMC.

    Parameters
    ----------
    model: pm.Model
        PyMC model object.
    draws : int
        Number of draws to keep from the sampling process.
    tune : int
        Number of tuning steps to take before sampling.
    chains : int
        Number of chains to sample.
    target_accept : float
        Target acceptance probability for step size adaptation.
    random_seed : int
        Seed for randomness.
    """
    with model:
        trace = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            cores=4,
            idata_kwargs={"log_likelihood": False}
        )
    
    try:
        az.to_netcdf(trace, path)
    except Exception as e:
        print(f"Error saving trace to {path}: {e}")
    return trace

def compute_log_likelihood(model: pm.Model, trace: az.InferenceData) -> None:
    """Wrapper to compute elemwise log_likelihood of model given InferenceData with posterior group
    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling
    """
    with model:
        pm.compute_log_likelihood(trace)
    return None

def sample_posterior_pred(model: pm.Model, trace: az.InferenceData) -> az.InferenceData:
    """Generates samples from the posterior predictive distribution for model checks

    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling

    Returns:
        az.InferenceData: An ArviZ InferenceData object containing the posterior predictive samples.
    """
    with model:
        spp = pm.sample_posterior_predictive(
            trace,
            extend_inferencedata=True,
            random_seed=RANDOM_SEED,
        )
    return spp

In [9]:
shot_type_idx_vals = df['shot_type_idx'].values
player_idx_vals = df['player_idx'].values
season_idx_vals = df['season_idx'].values
baseline_logit_vals = df['baseline_logit'].values
defensive_team_idx_vals = df['def_idx'].values
court_idx_vals = df['court_idx'].values

In [10]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":["JUMPER", "LAYUP", "HOOK", "DUNK"],
    "player": ["Average"]+pd.unique(df['PLAYER']).tolist(),
    "defensive_team": ["Average"]+pd.unique(df['defTeamTricode']).tolist(),
    "court": ["Average"]+pd.unique(df['homeTeamTricode']).tolist(),
    "season" : ['2022/2023','2023/2024','2024/2025'],
    "season_walk" : ['2023/2024','2024/2025'],
    }
with pm.Model(coords=coords) as model:
    # mutable data
    shot_made = pm.Data("shot_made", target, dims=("obs_id",))
    baseline_logits = pm.Data("baseline_logits", baseline_logit_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals.astype("int32"), dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals.astype("int32"), dims=("obs_id",))
    def_idx = pm.Data("def_idx", defensive_team_idx_vals.astype("int32"), dims=("obs_id",))
    court_idx = pm.Data("court_idx", court_idx_vals.astype("int32"), dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals.astype("int32"), dims=("obs_id",))
    
    # baseline
    baseline_weight = pm.Normal("baseline_weight", mu=1.0, sigma=0.25)

    # shot type effect
    sigma_shot_type = pm.HalfNormal("sigma_shot_type", 1.0)
    shot_type_effect = pm.Normal("shot_type_effect", mu=0.0, sigma=sigma_shot_type, dims=("shot_type",))
    shot_type_contribution = shot_type_effect[shot_type_idx]
    
    # career talent
    sigma_career = pm.HalfNormal("sigma_career", sigma=1.0, dims=("shot_type",))
    player_mean_dev = pm.Normal("player_mean_dev", mu=0.0, sigma=1.0, dims=("player", "shot_type"))
    player_mean_centered = player_mean_dev - player_mean_dev.mean(axis=0)
    player_mean = pm.Deterministic(
        "player_mean", 
        player_mean_centered * sigma_career[None, :], 
        dims=("player", "shot_type")
    )

    # walk
    sigma_walk = pm.HalfNormal("sigma_walk", sigma=1.0, dims=("shot_type",))
    player_innovations = pm.Normal(
        "player_innovations",
        mu=0.0, sigma=1.0,
        dims=("season_walk", "player", "shot_type")
    )
    raw_walk = pt.concatenate([
        pt.zeros((1, len(coords["player"]), len(coords["shot_type"]))),
        pt.cumsum(player_innovations, axis=0)
    ], axis=0)
    player_walk_centered = raw_walk - raw_walk.mean(axis=0, keepdims=True)
    player_walk = player_walk_centered * sigma_walk[None, None, :]
    # player effect
    player_effect = pm.Deterministic(
        "player_effect",
        player_mean[None, :, :] + player_walk,
        dims=("season", "player", "shot_type")
    )
    player_contribution = player_effect[season_idx, player_idx, shot_type_idx]
    
    # defensive team
    sigma_def = pm.HalfNormal("sigma_def", 1.0, dims=("shot_type",))
    def_offset = pm.Normal(
        "def_offset", mu=0.0, sigma=1.0,
        dims=("defensive_team", "season", "shot_type")
    )
    def_centered = def_offset - def_offset.mean(axis=0, keepdims=True)
    def_effect = pm.Deterministic(
        "def_effect",
        def_centered * sigma_def[None, None, :],
        dims=("defensive_team", "season", "shot_type")
    )
    def_contribution = def_effect[def_idx, season_idx, shot_type_idx]

    # court effect
    sigma_court = pm.HalfNormal("sigma_court", 1.0)
    court_raw = pm.Normal("court_raw", 0.0, sigma_court, dims=("court",))
    court_effect = pm.Deterministic("court_effect", court_raw - court_raw.mean(), dims=("court",))
    court_contribution = court_effect[court_idx]
    
    # additive components
    theta = baseline_weight*baseline_logits + shot_type_contribution + player_contribution + def_contribution + court_contribution
    shot_lkhood = pm.Bernoulli("shot_lkhood", logit_p=theta, observed=shot_made, dims=("obs_id",))

In [ ]:
trace = sample(model, tune=1500, draws=1500, target_accept=0.95, path='trace_objects/trace_hier_rw_tight.nc')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [baseline_weight, sigma_shot_type, shot_type_effect, sigma_career, player_mean_dev, sigma_walk, player_innovations, sigma_def, def_offset, sigma_court, court_raw]


Output()

Sampling 4 chains for 1_500 tune and 1_500 draw iterations (6_000 + 6_000 draws total) took 6889 seconds.


In [12]:
with open("models_pkl/xs_model_rw.pkl", "wb") as f:
    cloudpickle.dump(model, f)

In [15]:
trace = az.from_netcdf("trace_objects/trace_hier_rw_tight.nc")

In [16]:
az.summary(
    trace,
    var_names=[
        "baseline_weight",
        "sigma_shot_type",
        "shot_type_effect",
        "sigma_career",
        "sigma_walk",
        "sigma_def",
        "sigma_court"
        ]
)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
baseline_weight,1.119,0.011,1.098,1.138,0.000,0.000,8864.0,4242.0,1.00
sigma_shot_type,0.767,0.287,0.361,1.310,0.003,0.006,13254.0,3964.0,1.00
shot_type_effect[JUMPER],-0.000,0.008,-0.015,0.015,0.000,0.000,5611.0,4762.0,1.00
shot_type_effect[LAYUP],-0.276,0.010,-0.296,-0.258,0.000,0.000,7711.0,5323.0,1.00
shot_type_effect[HOOK],-0.129,0.028,-0.180,-0.076,0.000,0.000,5788.0,4846.0,1.00
shot_type_effect[DUNK],1.197,0.027,1.146,1.248,0.000,0.000,7656.0,4783.0,1.00
sigma_career[JUMPER],0.140,0.007,0.126,0.153,0.000,0.000,2056.0,3875.0,1.00
sigma_career[LAYUP],0.207,0.010,0.187,0.225,0.000,0.000,2530.0,4413.0,1.00
sigma_career[HOOK],0.323,0.031,0.266,0.382,0.001,0.000,2313.0,3633.0,1.00
sigma_career[DUNK],0.382,0.032,0.321,0.441,0.001,0.000,2501.0,3419.0,1.00


In [17]:
az.summary(
    trace,
    var_names=[
        "def_effect",
        ]
)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"def_effect[Average, 2022/2023, JUMPER]",-0.000,0.036,-0.068,0.067,0.000,0.001,15183.0,3904.0,1.0
"def_effect[Average, 2022/2023, LAYUP]",-0.014,0.103,-0.211,0.171,0.001,0.002,13755.0,4601.0,1.0
"def_effect[Average, 2022/2023, HOOK]",-0.006,0.052,-0.109,0.094,0.001,0.001,10254.0,4694.0,1.0
"def_effect[Average, 2022/2023, DUNK]",0.003,0.216,-0.401,0.403,0.002,0.003,14252.0,4070.0,1.0
"def_effect[Average, 2023/2024, JUMPER]",-0.006,0.034,-0.073,0.058,0.000,0.001,13956.0,4034.0,1.0
...,...,...,...,...,...,...,...,...,...
"def_effect[LAC, 2023/2024, DUNK]",0.211,0.146,-0.062,0.484,0.001,0.002,11904.0,3648.0,1.0
"def_effect[LAC, 2024/2025, JUMPER]",-0.028,0.025,-0.078,0.016,0.000,0.000,10963.0,4480.0,1.0
"def_effect[LAC, 2024/2025, LAYUP]",0.049,0.051,-0.048,0.143,0.000,0.001,16585.0,4127.0,1.0
"def_effect[LAC, 2024/2025, HOOK]",-0.007,0.049,-0.108,0.091,0.000,0.001,14260.0,4824.0,1.0


In [18]:
az.summary(
    trace,
    var_names=[
        "court_effect",
        ]
)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
court_effect[Average],-0.001,0.070,-0.134,0.132,0.001,0.001,8491.0,4023.0,1.0
court_effect[BOS],0.087,0.019,0.051,0.122,0.000,0.000,9207.0,4835.0,1.0
court_effect[GSW],0.116,0.019,0.081,0.150,0.000,0.000,7604.0,5000.0,1.0
court_effect[DET],0.057,0.019,0.021,0.092,0.000,0.000,9106.0,5258.0,1.0
court_effect[IND],0.063,0.019,0.025,0.097,0.000,0.000,9324.0,4811.0,1.0
court_effect[ATL],-0.078,0.019,-0.113,-0.044,0.000,0.000,8010.0,5123.0,1.0
court_effect[BKN],-0.035,0.019,-0.070,-0.001,0.000,0.000,8740.0,4770.0,1.0
court_effect[MIA],0.058,0.018,0.024,0.094,0.000,0.000,9757.0,5459.0,1.0
court_effect[TOR],-0.042,0.019,-0.076,-0.006,0.000,0.000,9919.0,4817.0,1.0
court_effect[MEM],-0.049,0.018,-0.083,-0.014,0.000,0.000,9630.0,5179.0,1.0


In [19]:
az.summary(
    trace,
    var_names=[
        "player_mean",
        ]
)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"player_mean[Average, JUMPER]",-0.001,0.136,-0.250,0.260,0.001,0.002,16100.0,4076.0,1.0
"player_mean[Average, LAYUP]",-0.001,0.209,-0.376,0.403,0.002,0.004,14472.0,3778.0,1.0
"player_mean[Average, HOOK]",0.000,0.321,-0.606,0.614,0.003,0.005,14537.0,4535.0,1.0
"player_mean[Average, DUNK]",0.001,0.380,-0.676,0.743,0.003,0.006,15086.0,4329.0,1.0
"player_mean[Joel Embiid, JUMPER]",0.233,0.055,0.126,0.333,0.000,0.001,12171.0,4379.0,1.0
...,...,...,...,...,...,...,...,...,...
"player_mean[Bobi Klintman, DUNK]",0.024,0.382,-0.675,0.744,0.003,0.006,16521.0,4536.0,1.0
"player_mean[Tolu Smith, JUMPER]",0.001,0.141,-0.258,0.267,0.001,0.002,15117.0,3673.0,1.0
"player_mean[Tolu Smith, LAYUP]",-0.000,0.198,-0.365,0.381,0.002,0.003,15708.0,4020.0,1.0
"player_mean[Tolu Smith, HOOK]",-0.001,0.323,-0.617,0.589,0.003,0.005,15649.0,3872.0,1.0


In [20]:
az.summary(
    trace,
    var_names=[
        "player_effect",
        ]
)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
"player_effect[2022/2023, Average, JUMPER]",-0.001,0.141,-0.275,0.260,0.001,0.002,16613.0,3649.0,1.0
"player_effect[2022/2023, Average, LAYUP]",-0.001,0.214,-0.423,0.383,0.002,0.004,15369.0,3914.0,1.0
"player_effect[2022/2023, Average, HOOK]",-0.000,0.325,-0.594,0.642,0.003,0.005,14140.0,4641.0,1.0
"player_effect[2022/2023, Average, DUNK]",0.002,0.393,-0.760,0.705,0.003,0.006,14841.0,4309.0,1.0
"player_effect[2022/2023, Joel Embiid, JUMPER]",0.255,0.058,0.141,0.359,0.001,0.001,11289.0,4279.0,1.0
...,...,...,...,...,...,...,...,...,...
"player_effect[2024/2025, Bobi Klintman, DUNK]",0.026,0.394,-0.711,0.745,0.003,0.006,15935.0,4453.0,1.0
"player_effect[2024/2025, Tolu Smith, JUMPER]",0.001,0.146,-0.256,0.285,0.001,0.002,15738.0,4199.0,1.0
"player_effect[2024/2025, Tolu Smith, LAYUP]",-0.001,0.202,-0.380,0.379,0.002,0.003,15449.0,4076.0,1.0
"player_effect[2024/2025, Tolu Smith, HOOK]",-0.001,0.329,-0.647,0.588,0.003,0.006,14635.0,3503.0,1.0


In [37]:
## Now with WIDER priors

In [23]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":["JUMPER", "LAYUP", "HOOK", "DUNK"],
    "player": ["Average"]+pd.unique(df['PLAYER']).tolist(),
    "defensive_team": ["Average"]+pd.unique(df['defTeamTricode']).tolist(),
    "court": ["Average"]+pd.unique(df['homeTeamTricode']).tolist(),
    "season" : ['2022/2023','2023/2024','2024/2025'],
    "season_walk" : ['2023/2024','2024/2025'],
    }
with pm.Model(coords=coords) as model_wide:
    # mutable data
    shot_made = pm.Data("shot_made", target, dims=("obs_id",))
    baseline_logits = pm.Data("baseline_logits", baseline_logit_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals.astype("int32"), dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals.astype("int32"), dims=("obs_id",))
    def_idx = pm.Data("def_idx", defensive_team_idx_vals.astype("int32"), dims=("obs_id",))
    court_idx = pm.Data("court_idx", court_idx_vals.astype("int32"), dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals.astype("int32"), dims=("obs_id",))
    
    # baseline
    baseline_weight = pm.Normal("baseline_weight", mu=1.0, sigma=0.25)

    # shot type effect
    sigma_shot_type = pm.HalfNormal("sigma_shot_type", 5.0)
    shot_type_effect = pm.Normal("shot_type_effect", mu=0.0, sigma=sigma_shot_type, dims=("shot_type",))
    shot_type_contribution = shot_type_effect[shot_type_idx]
    
    # career talent
    sigma_career = pm.HalfNormal("sigma_career", sigma=5.0, dims=("shot_type",))
    player_mean_dev = pm.Normal("player_mean_dev", mu=0.0, sigma=5.0, dims=("player", "shot_type"))
    player_mean_centered = player_mean_dev - player_mean_dev.mean(axis=0)
    player_mean = pm.Deterministic(
        "player_mean", 
        player_mean_centered * sigma_career[None, :], 
        dims=("player", "shot_type")
    )

    # walk
    sigma_walk = pm.HalfNormal("sigma_walk", sigma=5.0, dims=("shot_type",))
    player_innovations = pm.Normal(
        "player_innovations",
        mu=0.0, sigma=5.0,
        dims=("season_walk", "player", "shot_type")
    )
    raw_walk = pt.concatenate([
        pt.zeros((1, len(coords["player"]), len(coords["shot_type"]))),
        pt.cumsum(player_innovations, axis=0)
    ], axis=0)
    player_walk_centered = raw_walk - raw_walk.mean(axis=0, keepdims=True)
    player_walk = player_walk_centered * sigma_walk[None, None, :]
    # player effect
    player_effect = pm.Deterministic(
        "player_effect",
        player_mean[None, :, :] + player_walk,
        dims=("season", "player", "shot_type")
    )
    player_contribution = player_effect[season_idx, player_idx, shot_type_idx]
    
    # defensive team
    sigma_def = pm.HalfNormal("sigma_def", 5.0, dims=("shot_type",))
    def_offset = pm.Normal(
        "def_offset", mu=0.0, sigma=5.0,
        dims=("defensive_team", "season", "shot_type")
    )
    def_centered = def_offset - def_offset.mean(axis=0, keepdims=True)
    def_effect = pm.Deterministic(
        "def_effect",
        def_centered * sigma_def[None, None, :],
        dims=("defensive_team", "season", "shot_type")
    )
    def_contribution = def_effect[def_idx, season_idx, shot_type_idx]

    # court effect
    sigma_court = pm.HalfNormal("sigma_court", 1.0)
    court_raw = pm.Normal("court_raw", 0.0, sigma_court, dims=("court",))
    court_effect = pm.Deterministic("court_effect", court_raw - court_raw.mean(), dims=("court",))
    court_contribution = court_effect[court_idx]
    
    # additive components
    theta = baseline_weight*baseline_logits + shot_type_contribution + player_contribution + def_contribution + court_contribution
    shot_lkhood = pm.Bernoulli("shot_lkhood", logit_p=theta, observed=shot_made, dims=("obs_id",))

In [24]:
trace_wide = sample(model_wide, tune=1500, draws=1500, target_accept=0.95, path='trace_objects/trace_hier_rw_wide.nc')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [baseline_weight, sigma_shot_type, shot_type_effect, sigma_career, player_mean_dev, sigma_walk, player_innovations, sigma_def, def_offset, sigma_court, court_raw]


Output()

Sampling 4 chains for 1_500 tune and 1_500 draw iterations (6_000 + 6_000 draws total) took 6433 seconds.


In [25]:
with open("models_pkl/xs_model_wide.pkl", "wb") as f:
    cloudpickle.dump(model_wide, f)

In [26]:
trace_wide = az.from_netcdf('trace_objects/trace_hier_rw_wide.nc')

In [27]:

az.summary(
    trace_wide,
    var_names=[
        "baseline_weight",
        "sigma_shot_type",
        "shot_type_effect",
        "sigma_career",
        "sigma_walk",
        "sigma_def",
        "sigma_court"
        ]
)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
baseline_weight,1.119,0.011,1.098,1.139,0.000,0.000,9801.0,4990.0,1.00
sigma_shot_type,0.936,0.561,0.331,1.828,0.009,0.027,10031.0,3418.0,1.00
shot_type_effect[JUMPER],-0.000,0.008,-0.015,0.015,0.000,0.000,6204.0,5197.0,1.00
shot_type_effect[LAYUP],-0.276,0.010,-0.294,-0.257,0.000,0.000,7983.0,4952.0,1.00
shot_type_effect[HOOK],-0.129,0.028,-0.183,-0.079,0.000,0.000,5932.0,5529.0,1.00
shot_type_effect[DUNK],1.198,0.027,1.150,1.252,0.000,0.000,8123.0,5279.0,1.00
sigma_career[JUMPER],0.028,0.001,0.025,0.031,0.000,0.000,1851.0,3000.0,1.00
sigma_career[LAYUP],0.041,0.002,0.038,0.045,0.000,0.000,2520.0,4204.0,1.00
sigma_career[HOOK],0.065,0.006,0.053,0.076,0.000,0.000,2541.0,4030.0,1.00
sigma_career[DUNK],0.076,0.006,0.065,0.089,0.000,0.000,2612.0,3966.0,1.00


In [ ]:
## Now with EXP priors

In [29]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":["JUMPER", "LAYUP", "HOOK", "DUNK"],
    "player": ["Average"]+pd.unique(df['PLAYER']).tolist(),
    "defensive_team": ["Average"]+pd.unique(df['defTeamTricode']).tolist(),
    "court": ["Average"]+pd.unique(df['homeTeamTricode']).tolist(),
    "season" : ['2022/2023','2023/2024','2024/2025'],
    "season_walk" : ['2023/2024','2024/2025'],
    }
with pm.Model(coords=coords) as model_exp:
    # mutable data
    shot_made = pm.Data("shot_made", target, dims=("obs_id",))
    baseline_logits = pm.Data("baseline_logits", baseline_logit_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals.astype("int32"), dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals.astype("int32"), dims=("obs_id",))
    def_idx = pm.Data("def_idx", defensive_team_idx_vals.astype("int32"), dims=("obs_id",))
    court_idx = pm.Data("court_idx", court_idx_vals.astype("int32"), dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals.astype("int32"), dims=("obs_id",))
    
    # baseline
    baseline_weight = pm.Normal("baseline_weight", mu=1.0, sigma=0.25)

    # shot type effect
    sigma_shot_type = pm.Exponential("sigma_shot_type", 0.25)
    shot_type_effect = pm.Normal("shot_type_effect", mu=0.0, sigma=sigma_shot_type, dims=("shot_type",))
    shot_type_contribution = shot_type_effect[shot_type_idx]
    
    # career talent
    sigma_career = pm.Exponential("sigma_career", 0.25, dims=("shot_type",))
    player_mean_dev = pm.Normal("player_mean_dev", mu=0.0, sigma=3.0, dims=("player", "shot_type"))
    player_mean_centered = player_mean_dev - player_mean_dev.mean(axis=0)
    player_mean = pm.Deterministic(
        "player_mean", 
        player_mean_centered * sigma_career[None, :], 
        dims=("player", "shot_type")
    )

    # walk
    sigma_walk = pm.Exponential("sigma_walk", 0.25, dims=("shot_type",))
    player_innovations = pm.Normal(
        "player_innovations",
        mu=0.0, sigma=3.0,
        dims=("season_walk", "player", "shot_type")
    )
    raw_walk = pt.concatenate([
        pt.zeros((1, len(coords["player"]), len(coords["shot_type"]))),
        pt.cumsum(player_innovations, axis=0)
    ], axis=0)
    player_walk_centered = raw_walk - raw_walk.mean(axis=0, keepdims=True)
    player_walk = player_walk_centered * sigma_walk[None, None, :]
    # player effect
    player_effect = pm.Deterministic(
        "player_effect",
        player_mean[None, :, :] + player_walk,
        dims=("season", "player", "shot_type")
    )
    player_contribution = player_effect[season_idx, player_idx, shot_type_idx]
    
    # defensive team
    sigma_def = pm.Exponential("sigma_def", 0.25, dims=("shot_type",))
    def_offset = pm.Normal(
        "def_offset", mu=0.0, sigma=3.0,
        dims=("defensive_team", "season", "shot_type")
    )
    def_centered = def_offset - def_offset.mean(axis=0, keepdims=True)
    def_effect = pm.Deterministic(
        "def_effect",
        def_centered * sigma_def[None, None, :],
        dims=("defensive_team", "season", "shot_type")
    )
    def_contribution = def_effect[def_idx, season_idx, shot_type_idx]

    # court effect
    sigma_court = pm.HalfNormal("sigma_court", 1.0)
    court_raw = pm.Normal("court_raw", 0.0, sigma_court, dims=("court",))
    court_effect = pm.Deterministic("court_effect", court_raw - court_raw.mean(), dims=("court",))
    court_contribution = court_effect[court_idx]
    
    # additive components
    theta = baseline_weight*baseline_logits + shot_type_contribution + player_contribution + def_contribution + court_contribution
    shot_lkhood = pm.Bernoulli("shot_lkhood", logit_p=theta, observed=shot_made, dims=("obs_id",))

In [30]:
trace_exp = sample(model_exp, tune=1500, draws=1500, target_accept=0.95, path='trace_objects/trace_hier_rw_exp.nc')

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [baseline_weight, sigma_shot_type, shot_type_effect, sigma_career, player_mean_dev, sigma_walk, player_innovations, sigma_def, def_offset, sigma_court, court_raw]


Output()

Sampling 4 chains for 1_500 tune and 1_500 draw iterations (6_000 + 6_000 draws total) took 6549 seconds.


In [ ]:
with open("models_pkl/exp.pkl", "wb") as f:
    cloudpickle.dump(model_exp, f)

In [34]:
trace_exp = az.from_netcdf('trace_objects/trace_hier_rw_exp.nc')

In [35]:

az.summary(
    trace_exp,
    var_names=[
        "baseline_weight",
        "sigma_shot_type",
        "shot_type_effect",
        "sigma_career",
        "sigma_walk",
        "sigma_def",
        "sigma_court"
        ]
)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
baseline_weight,1.119,0.011,1.100,1.139,0.000,0.000,10333.0,4977.0,1.00
sigma_shot_type,0.894,0.517,0.324,1.719,0.007,0.026,10136.0,4163.0,1.00
shot_type_effect[JUMPER],-0.000,0.008,-0.015,0.014,0.000,0.000,5948.0,3816.0,1.00
shot_type_effect[LAYUP],-0.276,0.010,-0.296,-0.258,0.000,0.000,7442.0,4829.0,1.00
shot_type_effect[HOOK],-0.128,0.028,-0.180,-0.075,0.000,0.000,6598.0,4708.0,1.00
shot_type_effect[DUNK],1.197,0.027,1.144,1.248,0.000,0.000,8112.0,5012.0,1.00
sigma_career[JUMPER],0.047,0.002,0.042,0.051,0.000,0.000,2185.0,3502.0,1.00
sigma_career[LAYUP],0.069,0.003,0.063,0.075,0.000,0.000,2543.0,4074.0,1.00
sigma_career[HOOK],0.107,0.010,0.088,0.127,0.000,0.000,2469.0,3636.0,1.00
sigma_career[DUNK],0.128,0.010,0.107,0.147,0.000,0.000,2882.0,4264.0,1.00


In [ ]:
# get results

In [38]:
trace = az.from_netcdf("trace_objects/trace_hier_rw_tight.nc")

In [39]:
baseline_weight  = trace.posterior["baseline_weight"].values
shot_type_effect = trace.posterior["shot_type_effect"].values
player_effect = trace.posterior["player_effect"].values
player_mean = trace.posterior["player_mean"].values
def_effect = trace.posterior["def_effect"].values
court_effect = trace.posterior["court_effect"].values

In [40]:
shot_type_contribution = shot_type_effect[:, :, shot_type_idx_vals]

In [41]:
player_contribution = player_effect[:,:,season_idx_vals, player_idx_vals, shot_type_idx_vals]

In [42]:
def_contribution = def_effect[:, :, defensive_team_idx_vals, season_idx_vals, shot_type_idx_vals]

In [43]:
court_contribution = court_effect[:, :, court_idx_vals]

In [44]:
def compute_theta(baseline_weight, shot_type_contribution, player_contribution, def_contribution, court_contribution, batch_size=5000):
    n_obs = shot_type_contribution.shape[2]
    batches = np.arange(0, n_obs, batch_size)

    logits_mean_list = []
    logits_sd_list  = []
    probs_mean_list  = []

    for start in tqdm(batches):
        end = min(start + batch_size, n_obs)
        baseline_logit_batch = baseline_weight[:, :, None] * baseline_logit_vals[None, None, start:end]
        shot_type_contra_batch = shot_type_contribution[:, :, start:end]
        player_contra_batch = player_contribution[:, :, start:end]
        def_contra_batch = def_contribution[:, :, start:end]
        court_contra_batch = court_contribution[:, :, start:end]
        
        logits = baseline_logit_batch + shot_type_contra_batch + player_contra_batch + def_contra_batch + court_contra_batch
        probs = expit(logits)
        logits_mean = logits.mean(axis=(0, 1))
        logits_mean_list.append(logits_mean)
        logits_sd_list.append(logits.std(axis=(0, 1)))
        probs_mean_list.append(probs.mean(axis=(0, 1)))

        del logits, probs, baseline_logit_batch, shot_type_contra_batch, player_contra_batch, def_contra_batch, court_contra_batch

    return {
        "logits": np.concatenate(logits_mean_list),
        "logits_sd": np.concatenate(logits_sd_list),
        "probs": np.concatenate(probs_mean_list),
    }

In [45]:
results = compute_theta(
    baseline_weight,
    shot_type_contribution,
    player_contribution,
    def_contribution,
    court_contribution
    )

100%|██████████| 100/100 [00:50<00:00,  1.98it/s]


In [46]:
results["probs"].max(), results["probs"].min()

(np.float64(0.9732962919729016), np.float64(0.21496174774434404))

In [47]:
def compute_scores(target, preds):
    loss = log_loss(target, preds)
    auc = roc_auc_score(target, preds)
    binary_preds = (preds >= 0.5).astype(int)
    accuracy = accuracy_score(target, binary_preds)
    return loss, auc, accuracy

In [48]:
train_score = compute_scores(target, results['probs'])
print("Train Log Loss:", train_score[0])
print("Train AUC:", train_score[1])
print("Train Accuracy:", train_score[2])

Train Log Loss: 0.6404752729497863
Train AUC: 0.6586690319074555
Train Accuracy: 0.6284177742367726


In [49]:
df['probs_rw'] = results['probs']
df['logits_rw'] = results['logits']
df['logit_sd_rw'] = results['logits_sd']

In [50]:
df.to_csv('output/df_with_rw_preds.csv', index=False)

In [51]:
del trace, train_score, results, baseline_weight, shot_type_contribution, player_contribution, def_contribution, court_contribution

In [52]:
# get wide output
trace = az.from_netcdf("trace_objects/trace_hier_rw_wide.nc")

In [53]:
baseline_weight  = trace.posterior["baseline_weight"].values
shot_type_effect = trace.posterior["shot_type_effect"].values
player_effect = trace.posterior["player_effect"].values
player_mean = trace.posterior["player_mean"].values
def_effect = trace.posterior["def_effect"].values
court_effect = trace.posterior["court_effect"].values

In [54]:
shot_type_contribution = shot_type_effect[:, :, shot_type_idx_vals]

In [55]:
player_contribution = player_effect[:,:,season_idx_vals, player_idx_vals, shot_type_idx_vals]

In [58]:
def_contribution = def_effect[:, :, defensive_team_idx_vals, season_idx_vals, shot_type_idx_vals]

In [57]:
court_contribution = court_effect[:, :, court_idx_vals]

In [59]:
results = compute_theta(
    baseline_weight,
    shot_type_contribution,
    player_contribution,
    def_contribution,
    court_contribution
    )

100%|██████████| 100/100 [00:51<00:00,  1.92it/s]


In [60]:
train_score = compute_scores(target, results['probs'])
print("Train Log Loss:", train_score[0])
print("Train AUC:", train_score[1])
print("Train Accuracy:", train_score[2])

Train Log Loss: 0.6404734141657455
Train AUC: 0.6586681572279138
Train Accuracy: 0.6284177742367726


In [61]:
del trace, train_score, results, baseline_weight, shot_type_contribution, player_contribution, def_contribution, court_contribution

In [62]:
# get exp output
trace = az.from_netcdf("trace_objects/trace_hier_rw_exp.nc")

In [63]:
baseline_weight  = trace.posterior["baseline_weight"].values
shot_type_effect = trace.posterior["shot_type_effect"].values
player_effect = trace.posterior["player_effect"].values
player_mean = trace.posterior["player_mean"].values
def_effect = trace.posterior["def_effect"].values
court_effect = trace.posterior["court_effect"].values

In [64]:
shot_type_contribution = shot_type_effect[:, :, shot_type_idx_vals]

In [65]:
player_contribution = player_effect[:,:,season_idx_vals, player_idx_vals, shot_type_idx_vals]

In [66]:
def_contribution = def_effect[:, :, defensive_team_idx_vals, season_idx_vals, shot_type_idx_vals]

In [67]:
court_contribution = court_effect[:, :, court_idx_vals]

In [68]:
results = compute_theta(
    baseline_weight,
    shot_type_contribution,
    player_contribution,
    def_contribution,
    court_contribution
    )

100%|██████████| 100/100 [00:52<00:00,  1.89it/s]


In [69]:
train_score = compute_scores(target, results['probs'])
print("Train Log Loss:", train_score[0])
print("Train AUC:", train_score[1])
print("Train Accuracy:", train_score[2])

Train Log Loss: 0.6404780904522308
Train AUC: 0.6586639958082293
Train Accuracy: 0.6283896071496115


In [70]:
del trace, train_score, results, baseline_weight, shot_type_contribution, player_contribution, def_contribution, court_contribution